# V71 - Indonesian stemming

Pre-registered plan: chat is informal Indonesian with rich morphology, so menganalysis variants scatter across TF-IDF dimensions. Sastrawi stemming consolidates them before vectorizing. Identical word-only protocol and grouped folds under a fresh seed, gated against unstemmed word-only with the usual locks.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 313
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v71")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Stemmed vs raw word vectors, same folds

Stemming applies to the concatenated user document. Folds group by raw template as before, so the comparison isolates stemming.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
stemmer = StemmerFactory().create_stemmer()
sdocs = [stemmer.stem(d) for d in docs]
print("example:", docs[1][:80], "->", sdocs[1][:80])
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
WS = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(sdocs).toarray()
print(f"raw vocab dims={W.shape[1]} stemmed dims={WS.shape[1]}")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_raw, fid = ridge_oof(W[:n_train])
oof_stm, _ = ridge_oof(WS[:n_train])
base = ndcg5(Y, oof_raw)
d = ndcg5(Y, oof_stm) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_stm[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"raw={base.mean():.5f} stemmed={ndcg5(Y, oof_stm).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT stemming" if (lo > GATE_FLOOR and wf >= 0) else "KEEP raw tokens; stemming adds nothing")

example: Min aku mau fokus jadi data Analyst aja deh mau kuasain Excel sama SQL Eh min ga -> min aku mau fokus jadi data analyst aja deh mau kuasain excel sama sql eh min ga
raw vocab dims=319 stemmed dims=302


raw=0.62145 stemmed=0.61871 delta=-0.00274 CI95=[-0.00410,-0.00139] worst-fold=-0.00543
KEEP raw tokens; stemming adds nothing
